# Lesson 3.5 — 单帧策略 vs 历史策略（Partial Observability）

3.4 讲到 Diffusion Policy 用 $p_\theta(a\mid o)$ 处理**多峰动作**。本课要说明另一件容易与它混淆、但原因完全不同的事：**observation 丢失了信息**时，同一个 observation 会对应不同的正确动作。

对应 `docs/roadmap_v3.md` 的 3.6（单帧策略与历史策略）。

学完本课后应该能：

1. 说明 history 为什么能帮助 policy 推断**隐藏状态**；
2. 区分两种"多峰"：**action multimodality** 与 **partial observability**；
3. 用条件均值解释"为什么单帧 policy 会输出 0"；
4. 说出 frame stacking / RNN / Transformer 三种 history 机制的区别与代价；
5. 说明为什么长期任务最终需要 memory，而不只是一个更长的窗口。

## 1. 为什么 History 有用？

把 policy 从

$$
a_t=\pi(o_t)
$$

变成

$$
a_t=\pi(o_{t-k},\dots,o_t)
$$

这样 policy 可以从时间变化里**推断隐藏状态**。例如

```text
x(t-2) = -1
x(t-1) = -0.5
x(t)   =  0
```

模型可以推断出

$$
v>0
$$

即使 observation 没有直接给出 velocity。

所以 history 本质上是在帮助模型构造一个 **implicit state estimate**：

$$
\hat s_t=\phi(o_{t-k},\dots,o_t)
$$

这正是部分可观测（partially observable）问题的标准形式：真实状态 $s_t$ 看不到，只能从 observation 序列推断。

## 2. 制造一个 partial observation

下面用两条只有**速度符号相反**的轨迹，构造一个最小反例。设定是：

- 真实状态 $s=(x,\ v)$，其中 $x$ 是位置、$v$ 是速度；
- observation 只有 $o=x$；
- expert 的规则是"朝速度的反方向走"，即 $a^\star=-v$。

| | $x_{t-2}$ | $x_{t-1}$ | $x_t$ | 隐藏的 $v$ | expert $a^\star$ |
|---|---|---|---|---|---|
| Trajectory A | $-1.0$ | $-0.5$ | $0.0$ | $>0$ | $-1.0$ |
| Trajectory B | $+1.0$ | $+0.5$ | $0.0$ | $<0$ | $+1.0$ |

注意最后一行：两条轨迹在 $t$ 时刻的 **observation 完全相同**（都是 $x=0$），但正确的 action 相反。下面先构造这两条轨迹，再看单帧 policy 会遇到什么。

In [17]:
import numpy as np
import matplotlib.pyplot as plt

In [18]:
trajectory_a = np.array([
    [-1.0,  1.0],
    [-0.5,  1.0],
    [ 0.0,  1.0],
])

trajectory_b = np.array([
    [ 1.0, -1.0],
    [ 0.5, -1.0],
    [ 0.0, -1.0],
])

In [19]:
state_a = trajectory_a[-1]
state_b = trajectory_b[-1]

action_a = -state_a[1]
action_b = -state_b[1]

print("A state:", state_a, "expert action:", action_a)
print("B state:", state_b, "expert action:", action_b)

A state: [0. 1.] expert action: -1.0
B state: [ 0. -1.] expert action: 1.0


## 3. 单帧 policy 为什么会失败

如果单帧 policy 只能看到

$$
x=0
$$

那么训练数据里同时出现

$$
0\rightarrow-1 \qquad\text{和}\qquad 0\rightarrow+1
$$

MLP 会怎么办？如果使用 MSE，最优解不是"二选一"，而是**条件均值**。写成公式：设 $a^\star$ 在 $-1$ 与 $+1$ 上等概率，最小化

$$
\hat a=\arg\min_{\hat a}\ \mathbb E\big[(a^\star-\hat a)^2\big]
$$

的解是

$$
\hat a=\mathbb E[a^\star\mid o=0]=0,
\qquad \text{最小 MSE}=\operatorname{Var}(a^\star\mid o=0)=1
$$

也就是说：**MSE 最优的预测是一个什么都不做的 0 动作**，而在闭环里"什么都不做"通常意味着任务失败。注意这里有一个可测的后果——即使训练到完美，loss 也停在 $1$ 而不是 0，因为这部分误差是**信息缺失造成的**，不是模型不够大。

你有没有发现，这和刚才讲 Diffusion Policy 的 multimodal action 很像？

但**原因不同**。前面是：

> 同一个**状态**下真的存在多种合理动作。

现在是：

> 你以为是同一个状态，其实只是 **observation 丢失了信息**。

这是一个非常重要的区别：

$$
\boxed{\text{Action multimodality}}\qquad\text{vs}\qquad
\boxed{\text{Partial observability}}
$$

两者的数学形式可以写在一起看。设真实状态 $s$、observation $o$，则

$$
p(a\mid o)=\sum_{s} \underbrace{p(a\mid s)}_{\text{状态条件下的动作分布}}\;\underbrace{b(s)}_{\text{belief}=P(s\mid o_{1:t})}
$$

- **Action multimodality**：$p(a\mid s)$ 本身就是多峰的，即使状态完全已知；
- **Partial observability**：$p(a\mid s)$ 可能是单峰的，但 belief $b(s)$ 横跨了"应该做相反动作"的状态，于是 $p(a\mid o)$ 变成多峰。

**关键结论**：后一种多峰无法靠生成式模型解决——多采样几个动作不会让你知道哪个是对的。它只能靠**更多信息**（history / memory）或**主动感知**（移动一下去观察）解决。

## 4. History 如何解决？

现在不给 velocity，但提供两帧：

$$
(x_{t-1},\ x_t)
$$

Trajectory A：

```text
[-0.5, 0.0]
```

Trajectory B：

```text
[+0.5, 0.0]
```

虽然 $x_t$ 完全一样，但是 history 不一样，于是隐藏状态可以被估计出来：

$$
\hat v_t\approx\frac{x_t-x_{t-1}}{\Delta t}
$$

- A：$x_t-x_{t-1}=0-(-0.5)=+0.5$ ⟹ $\hat v>0$，与表中的隐藏速度符号一致；
- B：$x_t-x_{t-1}=0-(+0.5)=-0.5$ ⟹ $\hat v<0$。

所以 history 让"同一个 $x_t$"重新变得可区分。要留意两个工程含义：

1. **$\Delta t$ 必须已知且一致**：如果帧率变化或时间戳不同步，$\hat v$ 的尺度就错了（这正是本仓库 20 Hz / 50 Hz 时间契约问题的实际影响）；
2. **差分放大了噪声**：$x_{t-1}$ 与 $x_t$ 上的观测噪声会在差分里被放大 $1/\Delta t$ 倍，所以窗口越长不一定越好。

In [20]:
history_a = trajectory_a[-2:, 0]
history_b = trajectory_b[-2:, 0]

print("History A:", history_a)
print("History B:", history_b)

History A: [-0.5  0. ]
History B: [0.5 0. ]


## 5. Single-frame MLP vs History MLP

### 5.1 输入维度

**Single-frame MLP**

输入：

$$
x_t \qquad\text{shape }[B,1]
$$

**History MLP**：把过去 $K$ 帧直接拼起来

$$
[x_{t-K+1},\dots,x_t]\in\mathbb R^{K\cdot d_o}
$$

例如 $K=4$、$d_o=1$ 时是 $[B,4]$。然后

```text
history
   ↓
 flatten
   ↓
 MLP
   ↓
 action
```

这个方法叫 **Frame Stacking / Observation Stacking**。

### 5.2 为什么"flatten 之后 MLP"就够了

拼接保留了**顺序**：第 1 个位置是 $x_{t-K+1}$、最后一个是 $x_t$。因此 MLP 有能力在窗口上实现差分算子，例如

$$
\hat v\propto x_t-x_{t-1}=w^\top[x_{t-1},x_t]\quad\text{with } w=(-1,+1)
$$

这带来一个重要的判断方式：**当信息已经足够、而模型学不会时，问题在优化或容量；当信息本身缺失时，换更大的模型也没有用**。本课的反例属于后者。

### 5.3 固定窗口的代价

Frame stacking 是**有限记忆**：它只看最近 $K$ 帧，且输入维度随 $K$ 线性增长。对于"任务开始时发生了什么"这类信息（见 §7），$K$ 再大也覆盖不到，这时需要真正的 memory。

### 5.4 三种 history 机制的对比

| Policy | 输入 | Memory 形式 | 代价 |
|---|---|---|---|
| MLP | $o_t$ | 无 | 部分可观测下信息不足 |
| Stacked MLP | $o_{t-k:t}$ | 固定窗口 | 输入维度随 $k$ 增长；覆盖长度固定 |
| RNN / LSTM | sequential $o_t$ | hidden state $h_t=f(h_{t-1},o_t)$ | 压缩可能丢信息；训练较慢 |
| Transformer | sequence $o_{t-k:t}$ | 对 context 做 attention | 计算随窗口增长；需要位置编码 |

今天真正需要理解的不是哪一个"最好"，而是：

> 当当前 observation 不足以决定 action 时，policy **必须**获得某种形式的 context / memory。

换句话说：这是一个**信息问题**，架构只是获取信息的方式。

## 6. 从"信息是否足够"出发判断该用哪种 policy

把它们放在同一个视角下：policy 的输出应当依赖于**足够决定 action 的信息**。

$$
a_t=\pi\big(\underbrace{o_t}_{\text{当前观测}},\ \underbrace{m_t}_{\text{memory / belief}}\big),
\qquad m_t=\text{update}(m_{t-1},\ o_t,\ a_{t-1})
$$

- 若 $o_t$ 已经包含决策所需的全部信息，$m_t$ 可以省略（reactive policy）；
- 若 $o_t$ 不够（隐藏速度、遮挡、任务进度），$m_t$ 就不可省略，而 $m_t$ 的实现方式可以是固定窗口、RNN hidden state 或 attention。

诊断顺序建议：

1. 先问"**信息够不够**"（用本课这种最小反例就能验证：是否存在两个不同隐藏状态给出相同 observation、却需要不同 action）；
2. 再问"**表示够不够**"（history 拼进去后，同一反例是否变得可分）；
3. 最后才问"**容量/优化够不够**"（训练 loss 与验证 loss 是否还有下降空间）。

这个顺序能避免最常见的误判：把"信息缺失"当成"模型不够大"。

## 7. 从 reactive VLA 到带 memory 的 agent

Reactive VLA 的输入是当前时刻的观测与指令：

$$
(\text{image}_t,\ \text{state}_t,\ \text{language})\rightarrow \text{action}
$$

但长期任务不是这样。例如指令是：

> "把之前拿出来的杯子放回原来的柜子"

当前 image 里可能根本没有：

- "之前"发生了什么；
- "原来的柜子"是哪一个；
- "之前拿的是哪个杯子"。

也就是说，决定 action 所需的信息**不在当前观测里，而在历史里**。形式上，这是一个 belief / memory 依赖的策略：

$$
a_t=\pi\big(o_t,\ m_t\big),
\qquad m_t=\text{compress}(o_{1:t},\ a_{1:t-1})
$$

因此未来会变成：

$$
(\text{image}_t,\ \text{state}_t,\ \text{language},\ \text{memory})\rightarrow \text{action}
$$

这就是 roadmap 后面 **Embodied Memory & Task-State Tracking** 的基础：memory 不只是"更长的输入窗口"，而是要把历史压缩成**任务状态**（哪些步骤已完成、物体最后在哪里、是否失败过）。

一个诚实的边界：memory 解决的是"信息不在当前观测里"，**不是**解决数据不足或分布偏移。Lesson 2 的失败主要是后者（验证 MSE `0.2350` 劣于 baseline `0.1421`、held-out state 上 `|z|=13.21`、闭环 `0/10`），历史窗口并不会自动修好它。

## 小结（Part A：概念）

单帧 policy 使用

$$
a_t=\pi(o_t)
$$

它只在"当前 observation 足以决定 action"时成立。

在 partial observability 下，不同的隐藏状态会产生**相同的 observation**，于是：

1. 训练数据里同一个输入对应矛盾的目标；
2. MSE 的最优解是条件均值（本课例子中 $\hat a=0$），而不是任何一个正确动作；
3. 由此产生的多峰是**信息缺失**造成的，与 action multimodality（$p(a\mid s)$ 本身多峰）不是一回事：

$$
p(a\mid o)=\sum_s p(a\mid s)\,b(s)
$$

History-based policy 用

$$
a_t=\pi(o_{t-k:t})
$$

从时间上下文中恢复隐藏信息，常见实现是：

- observation stacking + MLP；
- RNN / LSTM；
- Transformer（attention over context）。

**判断标准**（本课真正要带走的）：先确认信息是否存在，再考虑表示，最后才考虑容量。

下面的 Part B 用一个合成数据集把这三条判断**实测**一次。

---

## 8. 实验：用合成的 partial-observation 数据集检验上面的判断

前面 §3 预测：单帧 policy 的 MSE 会停在**条件方差**而不是 0。下面构造一个能把"信息"隔离成唯一变量的数据集，训练两个结构完全相同、只改输入的 MLP，直接看这个预测是否成立。

### 8.1 数据生成：让"信息"成为唯一变量

真实状态：

$$
s_t=(x_t,\ v_t)
$$

但 policy 只能观察：

$$
o_t=x_t
$$

Expert 希望把运动停下来，因此：

$$
a_t=-v_t
$$

关键是：我们让当前位置 $x_t$ 和速度 $v_t$ **相互独立**（$x_t\sim U(-1,1)$，$v_t\in\{-1,+1\}$ 等概率）。这样同一个

$$
x_t=0.3
$$

可能对应

```text
v = +1 → action = -1
v = -1 → action = +1
```

单看当前 $x$ 根本没法知道正确答案。

数据里还额外存了一帧历史位置：

$$
x_{t-1}=x_t-v_t\,\Delta t
\qquad\Longleftrightarrow\qquad
v_t=\frac{x_t-x_{t-1}}{\Delta t}
$$

这个式子是整个实验的关键：它意味着**只要给 policy 两帧，隐藏速度就是精确可恢复的**（本例 $\Delta t=0.1$，所以 $v_t=10\,(x_t-x_{t-1})$）。于是"信息是否足够"成为唯一被改变的变量，模型容量、数据量、优化器都保持不变。

In [21]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

In [22]:
N = 4000
dt = 0.1

# Current position
x_t = np.random.uniform(-1.0, 1.0, size=N)

# Hidden velocity: either moving left or right
v_t = np.random.choice([-1.0, 1.0], size=N)

# Previous position
x_prev = x_t - v_t * dt

# Expert braking action
action = -v_t

### 8.2 两种输入表示

两种表示使用**完全相同的数据与标签**，只改变喂给 policy 的输入：

$$
X_\text{single}=[x_t]\in\mathbb R^{1}
\qquad\text{vs}\qquad
X_\text{history}=[x_{t-1},\ x_t]\in\mathbb R^{2}
$$

标签都是 expert 的刹车动作：

$$
Y=[a_t]=[-v_t]\in\mathbb R^{1}
$$

- **单帧输入**：$x_t$ 与 $v_t$ 独立，所以 $x_t$ 不包含决定 $a_t$ 的信息；
- **历史输入**：$v_t=(x_t-x_{t-1})/\Delta t$ 是两帧的确定性函数，信息完整。

这正是 §6 提出的诊断顺序里"第一步：信息是否存在"的可操作版本。

In [23]:
X_single = x_t[:, None].astype(np.float32)

In [24]:
X_history = np.stack(
    [x_prev, x_t],
    axis=1
).astype(np.float32)

Y = action[:, None].astype(np.float32)

In [25]:
print("Single:", X_single.shape)
print("History:", X_history.shape)
print("Action:", Y.shape)

Single: (4000, 1)
History: (4000, 2)
Action: (4000, 1)


### 8.3 训练协议：同一份数据、同一个模型、只改输入

- 数据量 $N=4000$，随机划分 $80\%$ 训练 / $20\%$ 验证（`indices = np.random.permutation(N)`）；
- 两个模型结构完全相同：`input_dim → 32 → 32 → 1`（`ReLU`），只有 `input_dim` 是 1 或 2；
- 相同的 `Adam(lr=1e-3)`、`MSE`、`batch_size=64`、50 个 epoch、固定 `seed=42`。

> 关于 split：这份合成数据的样本是 **i.i.d.** 的（每个 $x_t$ 独立采样），所以按样本随机划分是合法的。换成轨迹数据就不能这样——同一条轨迹的相邻帧高度相关，必须按 episode 划分（见 3.1 / 3.3 与 Lesson 2 的 2.8.7）。这也是本实验与真实数据集的第二个差别。

In [26]:
indices = np.random.permutation(N)

split = int(0.8 * N)

train_idx = indices[:split]
val_idx = indices[split:]

In [27]:
def make_loaders(X, Y, batch_size=64):

    X_train = torch.tensor(X[train_idx])
    Y_train = torch.tensor(Y[train_idx])

    X_val = torch.tensor(X[val_idx])
    Y_val = torch.tensor(Y[val_idx])

    train_dataset = TensorDataset(X_train, Y_train)
    val_dataset = TensorDataset(X_val, Y_val)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader

In [28]:
class MLPPolicy(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

In [29]:
single_model = MLPPolicy(input_dim=1)
history_model = MLPPolicy(input_dim=2)

In [30]:
def train_model(model, train_loader, val_loader, epochs=50):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    loss_fn = nn.MSELoss()

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # Training
        model.train()

        train_loss = 0.0

        for x, y in train_loader:

            optimizer.zero_grad()

            pred = model(x)

            loss = loss_fn(pred, y)

            loss.backward()

            optimizer.step()

            train_loss += loss.item() * len(x)

        train_loss /= len(train_loader.dataset)

        # Validation
        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for x, y in val_loader:

                pred = model(x)

                loss = loss_fn(pred, y)

                val_loss += loss.item() * len(x)

        val_loss /= len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

    return train_losses, val_losses

In [32]:
single_train, single_val = make_loaders(X_single, Y)
history_train, history_val = make_loaders(X_history, Y)

In [33]:
single_train_loss, single_val_loss = train_model(
    single_model,
    single_train,
    single_val
)

In [34]:
history_train_loss, history_val_loss = train_model(
    history_model,
    history_train,
    history_val
)

In [35]:
print(single_val_loss[-1])
print(history_val_loss[-1])

1.002676339149475
1.2975024510524236e-05


### 8.4 结果：`1.0027` 与 `1.3e-05`

两个模型在验证集上的最后一个 epoch MSE：

```text
single-frame : 1.002676339149475
history      : 1.2975024510524236e-05
```

这与 §3 的推导**定量吻合**：

- 单帧：$a=-v$，$v\in\{-1,+1\}$ 等概率且与 $x$ 独立，因此
  $\min \mathbb E[(a-\hat a)^2]=\operatorname{Var}(a)=1$，实测 `1.0027`；
- 历史：$v$ 是输入的确定性函数，任务退化为线性回归，理论上可达 $0$，实测 `1.3e-05`；
- 两者相差约 $7.7\times10^4$ 倍——**差别只来自输入里有没有那一帧 $x_{t-1}$**。

值得强调的是：单帧模型并不是"没训好"。它已经收敛到自己的**最优解**了，而这个最优解由信息量决定。

### 8.5 三种"不够"要分清

训练不收敛或效果差时，至少有三种完全不同的原因，它们的**症状不同、修法也不同**：

| 症状 | 诊断 | 正确做法 | 无效做法 |
|---|---|---|---|
| train loss 与 val loss 都高 | **模型容量 / 优化不足**（underfitting） | 增大模型、调学习率、换架构 | 加数据 |
| train loss 低、val loss 高 | **数据不足 / 方差大**（overfitting） | 加数据、正则、early stopping | 换更大的模型 |
| train 与 val 都停在某个**平台值**（本例约 `1`），换更大模型也不降 | **observation 信息不足** | 补 history / memory，或主动感知 | 加大模型、加数据、加训练轮数 |

第三种就是本实验的情形。它的判据可以写成一句话：如果这个平台值等于条件方差

$$
\min_{\hat a}\mathbb E\big[(a-\hat a)^2\mid o\big]=\operatorname{Var}(a\mid o)
$$

那么误差来自**信息**，而不是来自拟合能力。

### 8.6 More parameters ≠ more information

$$
\text{更多参数}\ \neq\ \text{更多信息}
$$

一个函数无论多大，都无法从 $o_t=x_t$ 中恢复 $v_t$——因为 $x_t$ 与 $v_t$ 在这份数据里是**独立的**：$p(v\mid x)=p(v)$。此时

$$
\mathbb E[a\mid x]=\mathbb E[-v\mid x]=-\mathbb E[v]=0
$$

所以单帧模型的**最优解就是输出 0**，任何容量都改变不了这个上界，只能更快地收敛到它。

反过来，把 $x_{t-1}$ 加进输入后，$v$ 变成输入的一个确定性函数，任务从"猜"变成"算"，误差上界随之从 $1$ 降到 $0$。这就是为什么"加输入"比"加参数"更根本——**先问信息，再问表示，最后才问容量**。

### 8.7 直接看模型输出：0 与 ±1

最后直接检查两个模型的行为。对单帧模型，取三个不同的位置；对历史模型，取两组"当前 $x$ 相同、上一帧 $x$ 相反"的输入（正是 §2 的最小反例）。

In [36]:
single_model.eval()

test_x = torch.tensor([
    [-0.5],
    [0.0],
    [0.5]
], dtype=torch.float32)

with torch.no_grad():
    pred = single_model(test_x)

print(pred)

tensor([[ 0.0521],
        [-0.0209],
        [-0.0270]])


In [37]:
history_model.eval()

examples = torch.tensor([
    [-0.1, 0.0],
    [ 0.1, 0.0]
], dtype=torch.float32)

with torch.no_grad():
    pred = history_model(examples)

print(pred)

tensor([[-0.9992],
        [ 1.0029]])


## 小结（Part B：实验）

这个合成实验把 Part A 的三条判断变成了可测的数字：

| | 输入 | 验证 MSE | 含义 |
|---|---|---|---|
| 单帧 MLP | $x_t$ | `1.0027` | 等于 $\operatorname{Var}(a\mid o)=1$，**信息上限** |
| History MLP | $[x_{t-1},x_t]$ | `1.3e-05` | 隐藏状态可精确恢复，误差趋 0 |

直接看输出更清楚：

- 单帧模型对 `[-0.5, 0.0, 0.5]` 都输出约 `0`——这正是条件均值 $\mathbb E[a\mid x]=0$，**不是训练失败，而是信息不足下的最优解**；
- 历史模型对 `[-0.1, 0.0]` 与 `[0.1, 0.0]` 输出 `-0.999` 与 `+1.003`——同样的 $x_t=0$，它靠 $x_{t-1}$ 恢复了速度符号。

结论：

1. **Partial observability 会造成"多峰"**，但原因是 $p(a\mid o)=\sum_s p(a\mid s)b(s)$ 里的 belief 混合，不是动作本身多峰；
2. 这种误差有**平台**，平台高度等于条件方差，因此可以被识别、也可以被预测；
3. 正确的修法是补信息（history / memory / 主动感知），而**不是**加参数或加数据；
4. 判断顺序永远是：信息 → 表示 → 容量。本实验对应第一步与第二步。

一个真实世界里的对应：这解释了为什么只给单帧状态的 policy 在部署时对"运动趋势"无能为力，也解释了本仓库后续为什么要做 memory 与 task-state tracking——那是把"不在当前 observation 里的信息"重新引入 policy 的工程形式。